<a href="https://colab.research.google.com/github/Zack2406/personal_portfolio/blob/main/Perceptron_and_Forward_Pass_Mansurov_Zohidjon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: The Perceptron, Activation Functions, and Forward Propagation

**Course:** Deep Learning and Neural Network (AIML302)
**Tools:** Python, NumPy, Matplotlib only

---

### Why we build things from scratch

In later labs we will use PyTorch, where a full neural network is a few lines of code.
Before that, it is worth seeing what those few lines actually do. Every modern network,
no matter how large, is built from one repeated operation: multiply the inputs by weights,
add a bias, and pass the result through a non-linear function. Today we write that
operation ourselves in NumPy.

### Objectives

By the end of this lab you should be able to:

1. Implement a single artificial neuron as a weighted sum plus a bias.
2. Implement and plot the common activation functions: step, sigmoid, tanh, ReLU.
3. Implement the perceptron learning rule and train it on a linearly separable problem.
4. Explain why a single perceptron cannot solve the XOR problem.
5. Implement the forward pass of a multi-layer network using matrix multiplication.
6. Visualize the decision boundary of any classifier in two dimensions.



---

## Part 1: Guided Walkthrough

## 1. Setup

We need only two libraries. NumPy gives us arrays and fast matrix operations,
Matplotlib gives us plots. We also fix the random seed so that every student's
notebook produces the same numbers, which makes it much easier to compare results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

print("NumPy version:", np.__version__)

## 2. The artificial neuron

A neuron takes a vector of inputs and produces a single number. It does this in two steps.

**Step 1 — the weighted sum.** Each input is multiplied by its own weight, the products are
added together, and a bias term is added at the end:

$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

**Step 2 — the activation.** The number `z` is passed through a non-linear function `f`:

$$a = f(z)$$

The weights decide how much each input matters. The bias shifts the result up or down,
which lets the neuron fire more easily or less easily regardless of the input.

Let us compute this by hand for a neuron with three inputs.

In [ ]:
x = np.array([1.0, 2.0, 3.0])      # the inputs
w = np.array([0.5, -1.0, 0.25])    # one weight per input
b = 2.0                            # a single bias

# The long way, to show what is happening
z_manual = x[0] * w[0] + x[1] * w[1] + x[2] * w[2] + b

# The short way: the dot product does exactly the same thing
z = np.dot(w, x) + b

print("weighted sum, written out :", z_manual)
print("weighted sum, dot product :", z)

Both lines give the same answer, but the dot product version is the one we will use
everywhere. It is shorter, and NumPy runs it far faster than a Python loop.

### Many inputs at once

In practice we do not push one example through the network at a time. We stack all our
examples into a matrix `X`, where **each row is one example** and each column is one feature.
A single matrix multiplication then computes the weighted sum for every example at once.

Watch the shapes carefully. Shape errors are the most common bug in neural network code,
and reading shapes is a skill worth building now.

In [ ]:
X = np.array([[1.0, 2.0, 3.0],     # example 1
              [0.0, 1.0, 1.0],     # example 2
              [2.0, 2.0, 0.0]])    # example 3

Z = X @ w + b      # the @ symbol is matrix multiplication

print("X shape:", X.shape)         # (3 examples, 3 features)
print("w shape:", w.shape)         # (3 weights,)
print("Z shape:", Z.shape)         # (3 outputs,) - one per example
print()
print("Z =", Z)

## 3. Activation functions

Without an activation function, a neuron is just a linear equation. Stacking many linear
equations still gives a linear equation, so a network of any depth would collapse into a
single line. The activation function is what breaks that linearity and lets networks learn
curved decision boundaries.

Here are the four you need to know today.

| Name | Formula | Output range | Notes |
|------|---------|--------------|-------|
| Step | 1 if z >= 0, else 0 | {0, 1} | Original perceptron. Not differentiable, so it cannot be trained by gradient descent. |
| Sigmoid | 1 / (1 + e^(-z)) | (0, 1) | Smooth. Output reads naturally as a probability. Saturates for large abs(z). |
| Tanh | (e^z - e^(-z)) / (e^z + e^(-z)) | (-1, 1) | Like sigmoid but centred on zero, which usually trains better. |
| ReLU | max(0, z) | [0, inf) | The default choice in modern networks. Cheap and does not saturate for positive z. |

Each one is a single line of NumPy.

In [ ]:
def step(z):
    """Step function: fires with 1 when z is non-negative, otherwise 0."""
    return np.where(z >= 0, 1.0, 0.0)


def sigmoid(z):
    """Sigmoid: squashes any real number into the range (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


def tanh(z):
    """Hyperbolic tangent: squashes any real number into the range (-1, 1)."""
    return np.tanh(z)


def relu(z):
    """ReLU: keeps positive values, sets negative values to zero."""
    return np.maximum(0.0, z)


# A quick check on a few values
test = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
print("input   :", test)
print("step    :", step(test))
print("sigmoid :", np.round(sigmoid(test), 3))
print("tanh    :", np.round(tanh(test), 3))
print("relu    :", relu(test))

Notice that each function was written to work on a whole array at once, not on a single
number. This is called vectorization, and it is why we used `np.where` and `np.maximum`
instead of a Python `if` statement. An `if` would fail on an array input.

Now let us look at their shapes.

In [ ]:
z_range = np.linspace(-6, 6, 300)

functions = [("Step", step), ("Sigmoid", sigmoid), ("Tanh", tanh), ("ReLU", relu)]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))

for ax, (name, fn) in zip(axes, functions):
    ax.plot(z_range, fn(z_range), color="black", linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.7)
    ax.axvline(0, color="grey", linewidth=0.7)
    ax.set_title(name)
    ax.set_xlabel("z")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("activation")
plt.tight_layout()
plt.show()

Two things to observe in these plots.

First, the step function jumps. It has no useful slope anywhere, which is exactly why the
original perceptron could not be trained with gradient descent and why we needed the smooth
alternatives.

Second, sigmoid and tanh flatten out at both ends. When a neuron's input lands far from zero,
the curve becomes almost horizontal, the gradient becomes almost zero, and learning slows to
a crawl. This is the vanishing gradient problem, and it is the main reason ReLU took over
in deep networks.

## 4. The perceptron as a classifier

A perceptron is one neuron with a step activation. It outputs 1 or 0, so it is a binary
classifier.

Let us use it on the logical AND function. There are four possible inputs, and the perceptron
should output 1 only when both inputs are 1.

We will not train anything yet. Instead we choose weights by hand and check that they work.

In [ ]:
X_and = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1], dtype=float)

# Chosen by hand: the sum must reach 1.5 before the neuron fires,
# which only happens when both inputs are 1.
w_and = np.array([1.0, 1.0])
b_and = -1.5


def perceptron_predict(X, w, b):
    """Forward pass of a perceptron: weighted sum, then step activation."""
    z = X @ w + b
    return step(z)


predictions = perceptron_predict(X_and, w_and, b_and)

print(" x1   x2   |    z    | predicted | target")
print("-" * 46)
for xi, zi, pi, ti in zip(X_and, X_and @ w_and + b_and, predictions, y_and):
    print(f"{xi[0]:4.0f} {xi[1]:4.0f}   | {zi:6.2f}  |     {pi:.0f}     |   {ti:.0f}")

All four predictions are correct.

### Where is the decision boundary?

The perceptron fires when `z >= 0`, so the dividing line between the two classes is the set
of points where `z` is exactly zero:

$$w_1 x_1 + w_2 x_2 + b = 0$$

That is the equation of a straight line. This is the single most important limitation of a
perceptron: **its decision boundary is always a straight line** (in higher dimensions, a flat
plane). It can only separate classes that a straight line can separate.

Rather than solve for the line algebraically, we will use a technique that works for any
classifier, however complicated. We cover the plane with a fine grid of points, classify
every point, and colour the result. The boundary appears by itself where the colours meet.

In [ ]:
def plot_decision_boundary(predict_fn, X, y, title, ax=None):
    """Colour the plane by predicted class, then draw the data points on top.

    predict_fn : function taking an (N, 2) array and returning N predictions
    X          : (N, 2) data points
    y          : (N,) true labels, 0 or 1
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4.5))

    # Step 1: build a grid covering the data, with a small margin
    pad = 0.5
    x1_min, x1_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    x2_min, x2_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                           np.linspace(x2_min, x2_max, 300))

    # Step 2: flatten the grid into an (N, 2) array of points and classify them all
    grid_points = np.c_[xx1.ravel(), xx2.ravel()]
    grid_pred = predict_fn(grid_points).reshape(xx1.shape)

    # Step 3: colour the grid, then draw the real data on top
    ax.contourf(xx1, xx2, grid_pred, levels=[-0.1, 0.5, 1.1],
                colors=["#d9d9d9", "#9a9a9a"], alpha=0.8)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="white", edgecolors="black",
               s=120, zorder=3, label="class 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="black", edgecolors="black",
               s=120, zorder=3, label="class 1")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    return ax


plot_decision_boundary(lambda G: perceptron_predict(G, w_and, b_and),
                       X_and, y_and, "AND with hand-chosen weights")
plt.tight_layout()
plt.show()

The line sits neatly between the single black point and the three white ones. Note that
many different lines would work here. Nothing makes this particular one special, and the
training algorithm we write next will find a different but equally valid line.

## 5. The perceptron learning rule

We chose those weights by hand, which does not scale. The perceptron learning rule finds
them automatically. It is a remarkably simple algorithm:

> For each training example, predict. If the prediction is correct, change nothing.
> If it is wrong, nudge the weights in the direction that would have fixed it.

The update is:

$$w \leftarrow w + \eta \, (y - \hat{y}) \, x, \qquad b \leftarrow b + \eta \, (y - \hat{y})$$

where `eta` is the learning rate. Look at what the error term `(y - y_hat)` does:

- Correct prediction: the error is 0, so nothing changes.
- Predicted 0 but the target was 1: the error is +1, so the weights move toward `x`, raising `z`.
- Predicted 1 but the target was 0: the error is -1, so the weights move away from `x`, lowering `z`.

One pass over the whole dataset is called an epoch.

In [ ]:
def perceptron_train(X, y, learning_rate=0.1, epochs=10, verbose=True):
    """Train a perceptron with the perceptron learning rule.

    Returns the learned weights, bias, and the error count per epoch.
    """
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0
    history = []

    for epoch in range(epochs):
        errors = 0
        for xi, target in zip(X, y):
            prediction = step(np.dot(w, xi) + b)
            error = target - prediction
            if error != 0:
                w = w + learning_rate * error * xi
                b = b + learning_rate * error
                errors += 1
        history.append(errors)
        if verbose:
            print(f"epoch {epoch + 1:2d} | misclassified: {errors} | "
                  f"w = {np.round(w, 2)} | b = {round(b, 2)}")

    return w, b, history


w_learned, b_learned, history = perceptron_train(X_and, y_and,
                                                 learning_rate=0.1, epochs=10)

The error count drops to zero and stays there. Once no example is misclassified, no update
happens, so the weights stop changing. The perceptron has converged.

Two useful facts. If the data can be separated by a straight line, this algorithm is
guaranteed to find such a line in a finite number of steps. If the data cannot be separated
by a straight line, the algorithm never settles and the error count keeps oscillating forever.
We are about to see that second case.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_decision_boundary(lambda G: perceptron_predict(G, w_learned, b_learned),
                       X_and, y_and, "AND, learned boundary", ax=axes[0])

axes[1].plot(range(1, len(history) + 1), history, marker="o", color="black")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("misclassified examples")
axes[1].set_title("Training error")
axes[1].set_ylim(-0.2, max(history) + 0.5)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. The XOR problem

XOR outputs 1 when the two inputs differ, and 0 when they are the same. It looks no harder
than AND. Let us train the same perceptron on it.

In [ ]:
X_xor = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

w_xor, b_xor, history_xor = perceptron_train(X_xor, y_xor,
                                             learning_rate=0.1, epochs=10)

The error count never reaches zero. Look at what it settles into: the updates made during
one pass cancel each other out, so the weights return to where they started and the next epoch
repeats exactly the same mistakes. The algorithm is stuck in a cycle. Running it for a thousand
more epochs would change nothing.

The reason becomes obvious as soon as we plot the four points. The two class-1 points sit on
one diagonal and the two class-0 points sit on the other. No straight line can put one
diagonal on one side and the other diagonal on the other side. Try drawing one by hand.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plot_decision_boundary(lambda G: perceptron_predict(G, w_xor, b_xor),
                       X_xor, y_xor, "XOR, perceptron fails", ax=axes[0])

axes[1].plot(range(1, len(history_xor) + 1), history_xor, marker="o", color="black")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("misclassified examples")
axes[1].set_title("Training error never reaches zero")
axes[1].set_ylim(-0.2, max(history_xor) + 0.5)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

This limitation was published by Minsky and Papert in 1969 and contributed to a long
decline in neural network research. The solution turned out to be straightforward: add a
layer of neurons in between.

## 7. Forward pass of a multi-layer network

A layer is a group of neurons that all look at the same inputs. Each neuron has its own
column of weights, so a layer's weights form a matrix.

For a layer with input matrix `X` of shape `(N, d_in)` and `d_out` neurons:

- `W` has shape `(d_in, d_out)`, one column per neuron
- `b` has shape `(d_out,)`, one bias per neuron
- the output is `f(X @ W + b)`, with shape `(N, d_out)`

A forward pass through the whole network means applying this to each layer in turn, feeding
each layer's output into the next.

In [ ]:
def dense_layer(X, W, b, activation):
    """Forward pass through one fully connected layer."""
    Z = X @ W + b
    return activation(Z)

### Solving XOR by hand

We will build a network with two inputs, a hidden layer of two neurons, and one output
neuron. Instead of training it, we set the weights by hand so we can see exactly how the
extra layer removes the limitation.

The idea is to make the hidden layer compute two easier functions, both of which a single
line can handle:

- hidden neuron 1 computes OR
- hidden neuron 2 computes AND

Then XOR is simply "OR is true, but AND is not", and that final combination is itself
linearly separable. The hidden layer has re-described the problem in a new set of
coordinates where a straight line is enough.

The large weight values below make the sigmoid behave almost like a step function, which
keeps the hidden outputs close to a clean 0 or 1.

In [ ]:
# Hidden layer: column 0 acts as OR, column 1 acts as AND
W1 = np.array([[20.0, 20.0],
               [20.0, 20.0]])
b1 = np.array([-10.0, -30.0])

# Output layer: fire when OR is on and AND is off
W2 = np.array([[ 20.0],
               [-20.0]])
b2 = np.array([-10.0])


def mlp_forward(X):
    """Forward pass: input -> hidden layer (sigmoid) -> output layer (sigmoid)."""
    H = dense_layer(X, W1, b1, sigmoid)      # (N, 2)
    out = dense_layer(H, W2, b2, sigmoid)    # (N, 1)
    return out.ravel()                        # flatten to (N,)


hidden = dense_layer(X_xor, W1, b1, sigmoid)
output = mlp_forward(X_xor)

print(" x1  x2 |  h1(OR)  h2(AND) |  output  | rounded | target")
print("-" * 60)
for xi, hi, oi, ti in zip(X_xor, hidden, output, y_xor):
    print(f"{xi[0]:3.0f} {xi[1]:3.0f} |  {hi[0]:5.2f}    {hi[1]:5.2f}   |  {oi:6.3f}  "
          f"|    {round(oi):.0f}    |   {ti:.0f}")

All four outputs are correct. The hidden column `h1` is 0 only for input (0, 0), which is
OR. The column `h2` is 1 only for input (1, 1), which is AND. The output layer combines them.

Now plot the decision boundary. Since the output is a probability rather than a hard 0 or 1,
we threshold it at 0.5 before plotting.

In [ ]:
plot_decision_boundary(lambda G: (mlp_forward(G) > 0.5).astype(float),
                       X_xor, y_xor, "XOR solved by a two-layer network")
plt.tight_layout()
plt.show()

The boundary is no longer a single straight line. It is made of two straight pieces that
together carve out a corner region, and that is enough to separate the diagonals.

This is the general principle behind depth. Each layer is still linear followed by a simple
non-linearity, but composing several of them produces boundaries of essentially any shape.
A network with one sufficiently large hidden layer can approximate any continuous function,
a result known as the universal approximation theorem.

## 8. A forward pass with random weights

One last point before the exercises. A forward pass works no matter what the weights are.
It does not know or care whether the weights are any good. Training is what makes them good,
and that is the subject of the next lab.

To make this concrete, here is a larger network with randomly initialized weights, run on a
dataset that is clearly not linearly separable.

In [ ]:
# Two concentric rings: an inner ring of class 0, an outer ring of class 1
n = 100
angles = np.random.uniform(0, 2 * np.pi, n)
r_inner = np.random.uniform(0.0, 1.2, n // 2)
r_outer = np.random.uniform(2.0, 3.0, n // 2)
radii = np.concatenate([r_inner, r_outer])

X_rings = np.c_[radii * np.cos(angles), radii * np.sin(angles)]
y_rings = np.concatenate([np.zeros(n // 2), np.ones(n // 2)])

# A network with 2 inputs, 8 hidden units, 1 output, all weights random
W1_r = np.random.randn(2, 8)
b1_r = np.zeros(8)
W2_r = np.random.randn(8, 1)
b2_r = np.zeros(1)


def random_net_forward(X):
    H = dense_layer(X, W1_r, b1_r, tanh)
    out = dense_layer(H, W2_r, b2_r, sigmoid)
    return out.ravel()


accuracy = np.mean((random_net_forward(X_rings) > 0.5) == y_rings)
print(f"Accuracy with random weights: {accuracy:.1%}")

plot_decision_boundary(lambda G: (random_net_forward(G) > 0.5).astype(float),
                       X_rings, y_rings, "Untrained network, random weights")
plt.tight_layout()
plt.show()

The accuracy is no better than guessing, and the boundary sits in an arbitrary place with
no relation to the rings. The architecture is perfectly capable of separating this data, and
we will train exactly this network to do so in a later lab. Nobody has told the weights what
the task is yet.

### Summary of Part 1

- A neuron is a weighted sum plus a bias, passed through an activation function.
- Vectorize with `X @ W + b` so that all examples are processed at once.
- Activations provide non-linearity. Without them, depth gains nothing.
- A perceptron's decision boundary is always a straight line, so XOR is out of reach.
- A hidden layer re-describes the input, and the boundary can then bend.
- A forward pass with untrained weights produces meaningless output. Training comes next.

Now continue to Part 2.

---

# Part 2: Graded Exercises

**Total: 100 marks**

### Instructions

1. Work on your own. You may look back at Part 1 as much as you like.
2. Write your code only where the comment `# YOUR CODE HERE` appears. Do not change the
   function names or the number of arguments, because the check cells depend on them.
3. Run the check cell after each question. A check that prints `PASS` means your function
   behaves correctly on the test cases. Marks are awarded for a correct and readable
   implementation, not only for a passing check.
4. Use NumPy operations rather than Python loops wherever you can. Loops are allowed where a
   question explicitly involves stepping through examples one at a time.
5. Answer the theory questions in Question 6 in the markdown cell provided, in your own words.
6. Save the notebook as `Lab2_YourName.ipynb` and submit before the end of the session.

### Mark distribution

| Question | Topic | Marks |
|----------|-------|-------|
| 1 | Activation functions from scratch | 20 |
| 2 | A single neuron | 10 |
| 3 | Perceptron on the OR gate | 20 |
| 4 | Two-layer forward pass | 20 |
| 5 | Theory questions | 30 |
| | **Total** | **100** |

Run the cell below before starting. It gives you a fresh set of imports so that Part 2 works
even if you restart the kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

print("Ready. Good luck.")

## Question 1: Activation functions from scratch (20 marks)

Implement four activation functions. Each must work on a NumPy array of any shape and return
an array of the same shape. Do not use a Python `if` statement, and do not call the matching
NumPy or SciPy helper where one exists (for example, write tanh from the exponential formula
rather than calling `np.tanh`).

**(a) `my_sigmoid(z)` — 5 marks**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**(b) `my_tanh(z)` — 5 marks**
$$\tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$$

**(c) `my_relu(z)` — 5 marks**
$$\text{ReLU}(z) = \max(0, z)$$

**(d) `my_leaky_relu(z, alpha=0.01)` — 5 marks**
$$\text{LeakyReLU}(z) = \begin{cases} z & \text{if } z > 0 \\ \alpha z & \text{otherwise} \end{cases}$$

Leaky ReLU is a small change to ReLU. Instead of setting negative inputs to exactly zero, it
lets a small fraction through. This keeps a non-zero gradient on the negative side, which
prevents a neuron from getting permanently stuck at zero output. `np.where` is useful here.

In [1]:
def my_sigmoid(z):
    """(a) Sigmoid activation. 5 marks."""
    # YOUR CODE HERE
    return 1.0 / (1.0 + np.exp(-z))


def my_tanh(z):
    """(b) Tanh activation, written from the exponential formula. 5 marks."""
    # YOUR CODE HERE
    exp_z = np.exp(z)
    exp_neg_z = np.exp(-z)
    return (exp_z - exp_neg_z) / (exp_z + exp_neg_z)


def my_relu(z):
    """(c) ReLU activation. 5 marks."""
    # YOUR CODE HERE
    return np.maximum(0.0, z)


def my_leaky_relu(z, alpha=0.01):
    """(d) Leaky ReLU activation. 5 marks."""
    # YOUR CODE HERE
    return np.where(z > 0, z, alpha * z)

In [3]:
import numpy as np

# CHECK CELL for Question 1 - do not modify
test_z = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])

checks = [
    ("my_sigmoid", my_sigmoid(test_z), 1 / (1 + np.exp(-test_z))),
    ("my_tanh", my_tanh(test_z), np.tanh(test_z)),
    ("my_relu", my_relu(test_z), np.maximum(0, test_z)),
    ("my_leaky_relu", my_leaky_relu(test_z), np.where(test_z > 0, test_z, 0.01 * test_z)),
]

for name, got, expected in checks:
    if got is None:
        print(f"FAIL {name}: function returned None")
    elif np.allclose(np.asarray(got, dtype=float), expected):
        print(f"PASS {name}")
    else:
        print(f"FAIL {name}: got {np.round(np.asarray(got, dtype=float), 4)}, "
              f"expected {np.round(expected, 4)}")

# Shape check: the functions must preserve the shape of a 2-D input
two_d = np.array([[-1.0, 2.0], [3.0, -4.0]])
if my_relu(two_d) is not None and np.asarray(my_relu(two_d)).shape == (2, 2):
    print("PASS shape preserved on 2-D input")
else:
    print("FAIL shape not preserved on 2-D input")

PASS my_sigmoid
PASS my_tanh
PASS my_relu
PASS my_leaky_relu
PASS shape preserved on 2-D input


Once your functions pass, run the cell below to plot them. Nothing to write here, but
check that each curve matches what you expect from the formula. The leaky ReLU plot uses an
exaggerated `alpha` so that the slope on the negative side is actually visible.

In [ ]:
z_range = np.linspace(-6, 6, 300)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
to_plot = [("Sigmoid", my_sigmoid(z_range)),
           ("Tanh", my_tanh(z_range)),
           ("ReLU", my_relu(z_range)),
           ("Leaky ReLU (alpha=0.1)", my_leaky_relu(z_range, alpha=0.1))]

for ax, (name, values) in zip(axes, to_plot):
    ax.plot(z_range, values, color="black", linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.7)
    ax.axvline(0, color="grey", linewidth=0.7)
    ax.set_title(name)
    ax.set_xlabel("z")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Question 2: A single neuron (10 marks)

**(a) `neuron_output(X, w, b, activation)` — 6 marks**

Write a function that computes the output of one neuron for a batch of examples. `X` has
shape `(N, d)`, `w` has shape `(d,)`, `b` is a single number, and `activation` is a function.
The result must have shape `(N,)`. Use a matrix operation, not a loop.

**(b) Apply it — 4 marks**

Using the data given in the code cell, call your function with `my_sigmoid` and store the
result in a variable named `q2_output`. Then print the shape of `q2_output` and its values
rounded to three decimal places.

In [4]:
def neuron_output(X, w, b, activation):
    """(a) Output of a single neuron for a batch of examples. 6 marks.

    X          : (N, d) input, one example per row
    w          : (d,) weights
    b          : float, bias
    activation : a function applied element-wise
    returns    : (N,) activations
    """
    # YOUR CODE HERE
    z = np.dot(X, w) + b
    return activation(z)


# (b) Apply your function to this data. 4 marks.
X_q2 = np.array([[1.0, 2.0],
                 [-1.0, 0.5],
                 [0.0, -3.0],
                 [2.0, 2.0]])
w_q2 = np.array([0.8, -0.4])
b_q2 = 0.1

q2_output = neuron_output(X_q2, w_q2, b_q2, my_sigmoid)   # YOUR CODE HERE: call neuron_output with my_sigmoid

# YOUR CODE HERE: print the shape of q2_output, and the values rounded to 3 decimals
print("Shape of q2_output:", q2_output.shape)
print("Values of q2_output (rounded to 3 decimals):")
print(np.round(q2_output, 3))

Shape of q2_output: (4,)
Values of q2_output (rounded to 3 decimals):
[0.525 0.289 0.786 0.711]


In [5]:
# CHECK CELL for Question 2 - do not modify
expected_q2 = 1 / (1 + np.exp(-(X_q2 @ w_q2 + b_q2)))

if q2_output is None:
    print("FAIL: q2_output is still None")
else:
    arr = np.asarray(q2_output, dtype=float)
    if arr.shape != (4,):
        print(f"FAIL: expected shape (4,), got {arr.shape}")
    elif np.allclose(arr, expected_q2):
        print("PASS neuron_output")
    else:
        print(f"FAIL: got {np.round(arr, 4)}, expected {np.round(expected_q2, 4)}")

PASS neuron_output


## Question 3: Perceptron on the OR gate (20 marks)

The OR gate outputs 1 when at least one input is 1. The truth table is given below.

**(a) `my_step(z)` and `perceptron_predict(X, w, b)` — 6 marks**

Write the step activation, then the perceptron forward pass that applies it to `X @ w + b`.

**(b) `train_perceptron(X, y, learning_rate, epochs)` — 10 marks**

Implement the perceptron learning rule. Start with weights and bias at zero. For each epoch,
loop over the examples one at a time. For each example compute the prediction, compute
`error = target - prediction`, and if the error is non-zero update both the weights and the
bias. Count how many examples were misclassified in the epoch and append that count to a list.

Return three things: the final weights, the final bias, and the list of error counts.

**(c) Train and report — 4 marks**

Train on the OR data with a learning rate of 0.1 for 10 epochs. Print the learned weights and
bias, print the epoch at which the error count first reached zero, and confirm that the
predictions match all four targets.

In [6]:
X_or = np.array([[0, 0],
                 [0, 1],
                 [1, 0],
                 [1, 1]], dtype=float)
y_or = np.array([0, 1, 1, 1], dtype=float)


def my_step(z):
    """(a) Step activation: 1 where z >= 0, otherwise 0. 3 marks."""
    # YOUR CODE HERE
    return np.where(z >= 0, 1.0, 0.0)


def perceptron_predict(X, w, b):
    """(a) Perceptron forward pass. 3 marks."""
    # YOUR CODE HERE
    z = np.dot(X, w) + b
    return my_step(z)

In [7]:
def train_perceptron(X, y, learning_rate=0.1, epochs=10):
    """(b) Train a perceptron with the perceptron learning rule. 10 marks.

    returns : w, b, history
              where history[i] is the number of misclassified examples in epoch i
    """
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0
    history = []

    for epoch in range(epochs):
        errors = 0
        for xi, target in zip(X, y):
            prediction = perceptron_predict(xi.reshape(1, -1), w, b)[0] # Reshape xi for perceptron_predict
            error = target - prediction
            if error != 0:
                w = w + learning_rate * error * xi
                b = b + learning_rate * error
                errors += 1
        history.append(errors)

    return w, b, history

In [8]:
# (c) Train and report. 4 marks.
# YOUR CODE HERE:
#   - train on X_or, y_or with learning_rate=0.1 and epochs=10
#   - store the results in w_or, b_or, history_or
#   - print the learned weights and bias
#   - print the first epoch number where the error count is 0
#   - print the predictions next to the targets

w_or, b_or, history_or = train_perceptron(X_or, y_or, learning_rate=0.1, epochs=10)

print(f"Learned weights (w_or): {w_or}")
print(f"Learned bias (b_or): {b_or}")

convergence_epoch = -1
for i, errors in enumerate(history_or):
    if errors == 0:
        convergence_epoch = i + 1
        break

if convergence_epoch != -1:
    print(f"Perceptron converged at epoch: {convergence_epoch}")
else:
    print("Perceptron did not converge within 10 epochs.")

predictions_or = perceptron_predict(X_or, w_or, b_or)
print("\nPredictions vs. Targets for OR gate:")
for i in range(len(X_or)):
    print(f"Input: {X_or[i]}, Predicted: {predictions_or[i]:.0f}, Target: {y_or[i]:.0f}")

Learned weights (w_or): [0.1 0.1]
Learned bias (b_or): -0.1
Perceptron converged at epoch: 4

Predictions vs. Targets for OR gate:
Input: [0. 0.], Predicted: 0, Target: 0
Input: [0. 1.], Predicted: 1, Target: 1
Input: [1. 0.], Predicted: 1, Target: 1
Input: [1. 1.], Predicted: 1, Target: 1


In [9]:
# CHECK CELL for Question 3 - do not modify
if w_or is None or history_or is None:
    print("FAIL: w_or / b_or / history_or are still None")
else:
    preds = perceptron_predict(X_or, w_or, b_or)
    if preds is not None and np.array_equal(np.asarray(preds, dtype=float), y_or):
        print("PASS all four OR outputs are correct")
    else:
        print(f"FAIL predictions: got {preds}, expected {y_or}")

    if len(history_or) == 10:
        print("PASS history has one entry per epoch")
    else:
        print(f"FAIL history length: got {len(history_or)}, expected 10")

    if history_or[-1] == 0:
        print("PASS training converged")
    else:
        print(f"FAIL: final epoch still had {history_or[-1]} errors")

PASS all four OR outputs are correct
PASS history has one entry per epoch
PASS training converged


## Question 4: Two-layer forward pass (20 marks)

Build the forward pass of a network with two inputs, a hidden layer of three neurons, and one
output neuron.

**(a) `dense_layer(X, W, b, activation)` — 8 marks**

One fully connected layer. `X` has shape `(N, d_in)`, `W` has shape `(d_in, d_out)`, `b` has
shape `(d_out,)`. Return the activated output with shape `(N, d_out)`.

**(b) `two_layer_forward(X, W1, b1, W2, b2)` — 8 marks**

Pass `X` through the hidden layer using `my_tanh`, then pass that result through the output
layer using `my_sigmoid`. Return a 1-D array of shape `(N,)`. The output of the second layer
has shape `(N, 1)`, so flatten it with `.ravel()` before returning.

**(c) Shape report — 4 marks**

Run your network on `X_q4` and print the shape of the input, the shape of the hidden layer
output, and the shape of the final output. Getting into the habit of checking shapes will save
you a great deal of debugging time later.

In [10]:
def dense_layer(X, W, b, activation):
    """(a) Forward pass through one fully connected layer. 8 marks."""
    # YOUR CODE HERE
    z = X @ W + b
    return activation(z)


def two_layer_forward(X, W1, b1, W2, b2):
    """(b) Full forward pass: tanh hidden layer, then sigmoid output. 8 marks.

    returns : (N,) array of outputs in the range (0, 1)
    """
    # YOUR CODE HERE
    hidden_layer_output = dense_layer(X, W1, b1, my_tanh)
    output_layer_output = dense_layer(hidden_layer_output, W2, b2, my_sigmoid)
    return output_layer_output.ravel()

In [11]:
# Fixed weights so that everyone gets the same answer
X_q4 = np.array([[0.0, 0.0],
                 [0.0, 1.0],
                 [1.0, 0.0],
                 [1.0, 1.0],
                 [0.5, 0.5]])

W1_q4 = np.array([[
    1.0, -2.0, 0.5],
                  [-1.5, 1.0, 2.0]])
b1_q4 = np.array([0.1, -0.2, 0.0])

W2_q4 = np.array([[
    1.5],
                  [-1.0],
                  [0.8]])
b2_q4 = np.array([-0.3])

# (c) Print the three shapes. 4 marks.
# YOUR CODE HERE
print(f"Input shape (X_q4): {X_q4.shape}")

hidden_output_q4 = dense_layer(X_q4, W1_q4, b1_q4, my_tanh)
print(f"Hidden layer output shape: {hidden_output_q4.shape}")

final_output_q4 = two_layer_forward(X_q4, W1_q4, b1_q4, W2_q4, b2_q4)
print(f"Final output shape: {final_output_q4.shape}")

Input shape (X_q4): (5, 2)
Hidden layer output shape: (5, 3)
Final output shape: (5,)


In [12]:
# CHECK CELL for Question 4 - do not modify
out_q4 = two_layer_forward(X_q4, W1_q4, b1_q4, W2_q4, b2_q4)

H_ref = my_tanh(X_q4 @ W1_q4 + b1_q4)
ref_q4 = (my_sigmoid(H_ref @ W2_q4 + b2_q4)).ravel()

if out_q4 is None:
    print("FAIL: two_layer_forward returned None")
else:
    arr = np.asarray(out_q4, dtype=float)
    if arr.shape != (5,):
        print(f"FAIL: expected shape (5,), got {arr.shape}")
    elif np.allclose(arr, ref_q4):
        print("PASS two_layer_forward")
        print("outputs:", np.round(arr, 4))
    else:
        print(f"FAIL: got {np.round(arr, 4)}, expected {np.round(ref_q4, 4)}")

PASS two_layer_forward
outputs: [0.5117 0.1793 0.9043 0.6798 0.6813]


## Question 5: Theory questions (15 marks)

Answer in your own words in the cell below. Two or three sentences each is enough. Answers
copied word for word from Part 1 will not receive full marks.

**(a) (3 marks)** A network has three layers and no activation functions at all, so each layer
computes only `X @ W + b`. Explain why this network can do no more than a single layer.

**(b) (3 marks)** The perceptron learning rule is guaranteed to converge on the AND gate but
never converges on XOR. State the property of the data that decides which of the two happens,
and explain how it relates to the shape of a perceptron's decision boundary.

**(c) (3 marks)** Sigmoid and tanh both flatten out for inputs far from zero. Explain what
this does to learning in a deep network, and name one activation function that reduces the
problem.

**(d) (3 marks)** In the XOR network of Part 1, hidden neuron 1 computed OR and hidden neuron 2
computed AND. Explain in general terms what job a hidden layer performs, using this example.

**(e) (3 marks)** You run a forward pass on an untrained network with random weights and get
roughly 50 percent accuracy on a two-class problem. Your classmate concludes that the network
architecture is too weak for the task. Explain why that conclusion does not follow.

### Your Answers

**(a)** If a network consists only of linear operations (matrix multiplication and bias addition), then stacking multiple such layers simply results in another single linear operation. In mathematical terms, $$(X \cdot W_1 + b_1) \cdot W_2 + b_2$$ simplifies to $$X \cdot (W_1 \cdot W_2) + (b_1 \cdot W_2 + b_2)$$, which effectively becomes $$X \cdot W_{new} + b_{new}$$. Activation functions introduce non-linearity, enabling the network to learn and represent complex, non-linear relationships in the data.

**(b)** The property of data that determines whether a perceptron will converge is its **linear separability**. For instance, the data for an AND gate is linearly separable (it can be perfectly divided by a straight line), while XOR gate data is not. Since a perceptron's decision boundary is always a straight line (or a hyperplane in higher dimensions), it fundamentally cannot classify data that isn't linearly separable.

**(c)** When activation functions like sigmoid and tanh become flat (saturate) for very large input values, their gradients approach zero. In deep networks, this leads to the **vanishing gradient problem**, where gradients backpropagating through many layers become extremely small, effectively halting learning in earlier layers. ReLU (Rectified Linear Unit) and its variants, such as Leaky ReLU, mitigate this by having a constant non-zero gradient for positive inputs, thus avoiding saturation on that side.

**(d)** A hidden layer transforms the raw input data into a new, more abstract representation. In the XOR example, the hidden layer converts the non-linearly separable XOR problem into a new feature space (the outputs of OR and AND), where the problem becomes linearly separable. This allows the subsequent output layer to easily classify the data. In essence, hidden layers learn meaningful features or combinations of features from the original inputs.

**(e)** Achieving approximately 50% accuracy with random weights in a two-class problem is an expected outcome and signifies performance no better than random guessing. This does not imply that the network's architecture is inherently too weak. The network needs to be **trained** using an appropriate learning algorithm (like backpropagation) to adjust its weights. Without training, the weights are arbitrary, and the network's potential to learn complex patterns remains untapped. The architecture itself might be perfectly capable, but it hasn't yet been taught what to do.